### Visualize Table 41 from Jurgens et al. 2024, which shows overlap between their study and Zheng et al. 2024

In [1]:
import pandas as pd
import numpy as np

In [2]:
GWAS_df = pd.read_csv("Jurgen_GWAS_DCM_csv_format.csv", header = 0)

In [3]:
GWAS_df.head()

,Locus #,FUMA nearest gene\n,Most highly prioritzed \ngene in locus,Prioritization \nscore,Tied gene in locus,Lead SNP,Chromosome,Position (GRCh37),GRCh37,Overlapping genome-wide significant locus\nin GWAS for DCM-Broad or DCM-Strict?,...,Locus #,Lead SNP,Top gene prioritized,Top gene points,Second gene prioritized,Third gene prioritized,Same gene\n(irrespective of score)?,Prioritized gene in Jurgens et al. \n(>=2.5 points; chosen in locus),"Prioritized gene in Zheng et al. \n(>=3 points, no ties)",Same gene\nIF both strongly prioritized?
0,1,C1orf86:AL590822.1:RP11-181G12.4,SKI,2.0,FAAP20,rs2503715,1,"2,144,107",1:2144107,False,...,1.0,rs2503715,SKI; PRKCZ; FAAP20; C1orf86,1.0,NaN,NaN,PARTIAL,False,False,NaN
1,2,RPL22,RNF207,3.0,NaN,rs11121483,1,"6,263,792",1:6263792,False,...,3.0,rs709209,RNF207,3.0,KLHL21,NaN,TRUE,True,True,True
2,3,HSPB7,HSPB7,2.0,CLCNKA; EPHA2,rs1763605,1,"16,338,925",1:16338925,True,...,4.0,rs1763604; rs945418,HSPB7,4.0,CLCNKA,CLCNKB; ZBTB17,TRUE,False,True,NaN
3,4,AKR1A1,MAST2,2.0,NaN,rs518365,1,"46,010,652",1:46010652,False,...,5.0,rs2993263,MAST2,2.0,AKRA1; MUTYH,NaN,TRUE,False,False,NaN
4,5,TTC39A,CDKN2C,1.5,TTC39A,rs72694111,1,"51,793,258",1:51793258,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN,NaN


In [4]:
GWAS_df.columns

Index(['Locus #', 'FUMA nearest gene\n',
       'Most highly prioritzed \ngene in locus', 'Prioritization \nscore',
       'Tied gene in locus', 'Lead SNP', 'Chromosome', 'Position (GRCh37)',
       'GRCh37',
       'Overlapping genome-wide significant locus\nin GWAS for DCM-Broad or DCM-Strict?',
       'Overlapping genome-wide significant locus\nin MTAG?', 'Locus # ',
       'Lead SNP ', 'Top gene prioritized', 'Top gene points',
       'Second gene prioritized', 'Third gene prioritized',
       'Same gene\n(irrespective of score)?',
       'Prioritized gene in Jurgens et al. \n(>=2.5 points; chosen in locus)',
       'Prioritized gene in Zheng et al. \n(>=3 points, no ties)',
       'Same gene\nIF both strongly prioritized?'],
      dtype='object')

#### Get the location of these SNPs, which are in GRCh37 and the prioritized gene. Save this as a bed file for liftover

In [5]:
GWAS_bed_df = GWAS_df[["Chromosome", "Position (GRCh37)", "Lead SNP", "Most highly prioritzed \ngene in locus", "Prioritization \nscore"]]
GWAS_bed_df.head()

,Chromosome,Position (GRCh37),Lead SNP,Most highly prioritzed \ngene in locus,Prioritization \nscore
0,1,"2,144,107",rs2503715,SKI,2.0
1,1,"6,263,792",rs11121483,RNF207,3.0
2,1,"16,338,925",rs1763605,HSPB7,2.0
3,1,"46,010,652",rs518365,MAST2,2.0
4,1,"51,793,258",rs72694111,CDKN2C,1.5


In [6]:
GWAS_bed_df = GWAS_bed_df.rename(columns = {"Chromosome": "chr", 
                                           "Position (GRCh37)": "start", 
                                            "Lead SNP": "rsID",
                                           "Most highly prioritzed \ngene in locus": "candidate_gene",
                                           "Prioritization \nscore": "total_score"})

In [7]:
GWAS_bed_df['end'] = GWAS_bed_df['start']
GWAS_bed_df = GWAS_bed_df[["chr", "start", "end", "rsID", "candidate_gene", "total_score"]]
GWAS_bed_df

,chr,start,end,rsID,candidate_gene,total_score
0,1,"2,144,107","2,144,107",rs2503715,SKI,2.0
1,1,"6,263,792","6,263,792",rs11121483,RNF207,3.0
2,1,"16,338,925","16,338,925",rs1763605,HSPB7,2.0
3,1,"46,010,652","46,010,652",rs518365,MAST2,2.0
4,1,"51,793,258","51,793,258",rs72694111,CDKN2C,1.5
...,...,...,...,...,...,...
60,19,"46,358,957","46,358,957",rs113996837,SYMPK,2.0
61,20,"33,590,358","33,590,358",rs4616,GSS,1.5
62,21,"30,530,131","30,530,131",rs62222424,BACH1,2.5
63,21,"40,644,170","40,644,170",rs8134638,BRWD1,2.0


### Reformat properly for `liftover`

In [8]:
GWAS_bed_df['chr'] = "chr" + GWAS_bed_df['chr'].astype(str)

In [9]:
GWAS_bed_df["start"] = GWAS_bed_df["start"].str.replace(",", "").astype(int)
GWAS_bed_df["end"] = GWAS_bed_df["end"].str.replace(",", "").astype(int)

In [10]:
GWAS_bed_df.head()

,chr,start,end,rsID,candidate_gene,total_score
0,chr1,2144107,2144107,rs2503715,SKI,2.0
1,chr1,6263792,6263792,rs11121483,RNF207,3.0
2,chr1,16338925,16338925,rs1763605,HSPB7,2.0
3,chr1,46010652,46010652,rs518365,MAST2,2.0
4,chr1,51793258,51793258,rs72694111,CDKN2C,1.5


In [11]:
GWAS_bed_df.shape

(65, 6)

### Save to csv, install liftover to liftover from hg19 to hg38

- conda install bioconda::ucsc-liftover

In [12]:
# don't include header, since liftOver doesn't want this
GWAS_bed_df.to_csv("Jurgens_GWAS_hits_GRCh37.bed", sep="\t", index=False, header=False)